# Type-Safe Multi-Agent Team

This notebook walks through the course's core pattern: typed delegation, MCP-style tool governance, shared memory, and a bounded self-repair loop.

The teaching lens is the production agentic lifecycle: Perception, Reasoning, Execution, Learning, and Governance. A single agent can attempt all five, but a managed team separates them into roles with explicit contracts and permissions.


## 1. Load the core implementation

The implementation runs offline. In production, keep the same schemas and replace the local teaching MCP adapter with the official MCP Python SDK transport.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

from enterprise_agent_team import (
    AgentIdentity,
    AgentRole,
    CodePatch,
    LocalTrainingMcpGateway,
    SharedAgenticMemory,
    ToolName,
    ToolPolicy,
    ToolRequest,
    TrustTier,
    demonstrate_schema_enforcement,
    run_virtual_software_company,
)


## 2. Run the full team

Watch for the lifecycle: the PM perceives the issue, the router reasons about worker choice, the coder and QA execute, QA feedback becomes learning memory, and the reviewer decides whether the system may ship.


In [ ]:
state = run_virtual_software_company('Create a safe add function for a shared utility module.')
state.model_dump(mode='json')


## 2.1 Map the Agentic Lifecycle

After the run, inspect the state as a lifecycle trace. This is the mental model learners should carry into the capstone `run_company(issue, max_repairs=2)` implementation.

In [ ]:
from pprint import pprint

lifecycle_trace = {
    "Perception (PM)": state.task.goal if state.task else None,
    "Reasoning (Router/Tech Lead)": [decision.model_dump(mode="json") for decision in state.route_trace],
    "Execution (Coder/QA)": {
        "patch_rationale": state.patch.rationale if state.patch else None,
        "tests_passed": state.test_result.passed if state.test_result else None,
        "test_log": state.test_result.log if state.test_result else None,
    },
    "Learning (Repair Loop)": {
        "repair_attempts": state.repair_attempts,
        "review_reasons": state.review.reasons if state.review else [],
    },
    "Governance (Review/Release)": state.review.model_dump(mode="json") if state.review else None,
}

pprint(lifecycle_trace)

## 3. Test schema enforcement

The `CodePatch` contract forbids extra fields and unsafe file paths. This is the guardrail that prevents raw agent text from becoming executable system behavior.

In [ ]:
demonstrate_schema_enforcement()


## 4. Exercise 4: Trust & Governance Matrix

A coder can search memory and write patches, but cannot execute QA tools. Authorization is checked on every call. For the learner assignment, extend this idea into a full zero-trust MCP-style gateway with identities, role rules, sensitivity tiers, and an audit log.


In [ ]:
memory = SharedAgenticMemory()
gateway = LocalTrainingMcpGateway(memory=memory, policy=ToolPolicy())

coder = AgentIdentity(
    agent_id='coder-unauthorized-demo',
    role=AgentRole.CODER,
    org_id='training',
    scopes={ToolName.SEARCH_MEMORY},
    clearance=TrustTier.CONFIDENTIAL,
)

request = ToolRequest(
    caller=coder,
    tool=ToolName.EXECUTE_TESTS,
    args={'patch': {'files': [], 'rationale': 'demo', 'tests_to_run': []}},
    data_sensitivity=TrustTier.CONFIDENTIAL,
)

try:
    gateway.call_tool(request)
except PermissionError as exc:
    print('Blocked:', exc)


## 5. Capstone Assignment

Implement `run_company(issue: str, max_repairs: int = 2)` in the capstone starter so it orchestrates:

1. Perception (PM): ingest the issue and extract requirements.
2. Reasoning (Tech Lead): decompose requirements and plan execution.
3. Execution (Coder/QA): write code, run tests, and gather feedback.
4. Learning (Repair Loop): evaluate failures and refine via memory.
5. Governance (Security/Release): enforce compliance before final output.

Modify the task, add a second tool, or change the retry budget. The design goal is always the same: every agent can act, but only through governed, typed boundaries.
